# 10 — Calibração e Escolha de Limiar

Calibra os finalistas e escolhe o limiar operacional usando **somente treino e validação** (`val_ref`). Com `HONEST_CALIB_SPLIT = True`, o `val_ref` é dividido em duas metades estratificadas e disjuntas: `val_cal` ajusta o calibrador e `val_meas` mede o Brier e escolhe o limiar. O finalista é o congelado em `selecao_modelos.finalista_escolhido` e o limiar segue `limiar.metrica` e `limiar.min_recall` do YAML. Não abre o teste e não escreve no YAML.

**Pré-requisito:** `pip install -e .` com scikit-learn ≥ 1.6.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split


def encontrar_raiz_projeto(start: Path | None = None) -> Path:
    atual = (start or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "src").is_dir() and (candidato / "pyproject.toml").is_file():
            return candidato
    raise RuntimeError(f"Raiz do projeto não encontrada a partir de: {atual}")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

try:
    import sklearn.frozen  # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "Este notebook requer scikit-learn >= 1.6 (FrozenEstimator)."
    ) from exc

from src import config as C
from src import config as env
from src.registro import Run
from src.dados import load_supervised_artifact, split_partitions
from src.avaliacao import check_leakage
from src.avaliacao import metricas_completas, brier
from src.avaliacao import curva_confiabilidade, ece, mce, brier_decomposicao
from src.modelos import build_finalista
from src.modelos import calibrar, escolher_limiar

env.carregar_dotenv()
env.garantir_diretorios()

TABLES_DIR      = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

SEED      = int(C.SEED)
G7_STATUS = str(C.cfg("gates", "G7_calibracao", "status") or "pendente")
G8_STATUS = str(C.cfg("gates", "G8_limiar", "status") or "pendente")

print(C.resumo())

ROOT_DIR          = <REPO>
DATA_DIR          = <REPO>\data
RAW_DIR           = <REPO>\data\raw
PROCESSED_DIR     = <REPO>\data\processed
ANALYTICAL_DIR    = <REPO>\data\analytical
TREINO_DIR        = <REPO>\data\treino
OUTPUTS_DIR       = <REPO>\outputs
RUNS_DIR          = <REPO>\outputs\runs
LOGS_DIR          = <REPO>\logs
CONFIGS_DIR       = <REPO>\configs
SUPERVISED_YAML   = <REPO>\configs\supervised.yaml (ok)
PARQUET_NAME      = srag.parquet
DATA_VERSION      = dados-v2
GITHUB_REPO       = Ivan-S-Oliveira/Projeto_Integrador
HTTP_TIMEOUT      = 180
HTTP_TENTATIVAS   = 3
SEED              = 42
GITHUB_TOKEN?     = False
SRAG_PARQUET_URL  = https://github.com/Ivan-S-Oliveira/Projeto_Integrador/releases/download/dados-v2/srag.parquet
SUPERVISED_SHA256 = ad64a9f7f65dbcf3b87c1f498e340c370eae1d723bd88f32a35f91133d4bb2f3


## Carregar artefacto e preparar validação

Se `HONEST_CALIB_SPLIT = True`, `val_cal` ajusta o calibrador e
`val_meas` mede — evita o viés otimista de calibrar e medir no mesmo
conjunto. Caso contrário, ambos são a validação inteira.

In [2]:
COHORT_KIND = os.getenv("SRAG_COHORT_KIND", "final").lower()

df, manifest = load_supervised_artifact(
    ROOT_DIR,
    expected_kind=COHORT_KIND,
)
treino, validacao = split_partitions(df, manifest)

PARTICAO = manifest["partition_column"]
ALVO     = manifest["target_column"]
FEATURES = list(manifest["feature_columns"])

X_train = treino[FEATURES].copy()
y_train = pd.to_numeric(treino[ALVO], errors="coerce").astype("int8")
X_val   = validacao[FEATURES].copy()
y_val   = pd.to_numeric(validacao[ALVO], errors="coerce").astype("int8")

check_leakage(X_train)
check_leakage(X_val)

HONEST_CALIB_SPLIT = True
CALIB_SPLIT_FRAC   = 0.5
CALIB_SPLIT_SEED   = SEED

if HONEST_CALIB_SPLIT:
    idx_cal, idx_meas = train_test_split(
        np.arange(len(y_val)),
        test_size=1.0 - CALIB_SPLIT_FRAC,
        random_state=CALIB_SPLIT_SEED,
        stratify=y_val.to_numpy(),
    )
    X_val_cal  = X_val.iloc[idx_cal].reset_index(drop=True)
    y_val_cal  = y_val.iloc[idx_cal].reset_index(drop=True)
    X_val_meas = X_val.iloc[idx_meas].reset_index(drop=True)
    y_val_meas = y_val.iloc[idx_meas].reset_index(drop=True)
    CALIB_SPLIT_HASH = hashlib.sha256(
        np.asarray(idx_cal, dtype=np.int64).tobytes()
    ).hexdigest()[:16]
else:
    X_val_cal, y_val_cal = X_val, y_val
    X_val_meas, y_val_meas = X_val, y_val
    CALIB_SPLIT_HASH = None

n_teste = int(sum(int(v) for k, v in manifest["partition_counts"].items() if str(k) in {"teste", "test_ref", "test_pandemic"}))

display(pd.DataFrame({
    "bloco":      ["treino", "val_cal", "val_meas"],
    "n":          [len(X_train), len(X_val_cal), len(X_val_meas)],
    "positivos":  [int(y_train.sum()), int(y_val_cal.sum()), int(y_val_meas.sum())],
}))

print(f"cohort_kind : {manifest['cohort_kind']}")
print(f"features    : {len(FEATURES)}")
print(f"teste       : {n_teste:,} (fechado)")
if CALIB_SPLIT_HASH:
    print(f"split_hash  : {CALIB_SPLIT_HASH}")

,bloco,n,positivos
0,treino,556462,46961
1,val_cal,59621,5032
2,val_meas,59622,5032


cohort_kind : final
features    : 30
teste       : 3,035,074 (fechado)
split_hash  : 9b621e331d8420e1


## Calibrar finalistas e escolher o limiar

Para cada finalista de `selecao_modelos.finalistas`, o pipeline é treinado
exclusivamente no conjunto de treino.

Os métodos `sigmoid` e `isotonic` são ajustados em `val_cal` e comparados pelo
Brier calculado em `val_meas`. Em caso de diferença de Brier inferior a
0,0005, o método `sigmoid` é escolhido por simplicidade e estabilidade.

Depois da escolha do calibrador:

1. o limiar operacional é selecionado em `val_cal`;
2. a regra é maximizar a precisão entre os cortes com recall mínimo definido
   em `limiar.min_recall`;
3. esse mesmo limiar é aplicado em `val_meas`;
4. nenhum reajuste é realizado em `val_meas`.

Os nomes do YAML são resolvidos em `src.modelos`, incluindo o alias
`hist_gradient_boosting -> boosting`.

O teste final permanece fechado durante toda esta execução.

In [3]:
selecao_cfg = C.cfg("selecao_modelos") or {}

FINALISTAS = list(selecao_cfg.get("finalistas") or [])
if not FINALISTAS:
    raise RuntimeError(
        "selecao_modelos.finalistas está vazio no YAML."
    )

FINALISTA_ESCOLHIDO = selecao_cfg.get("finalista_escolhido")
if FINALISTA_ESCOLHIDO is None:
    raise RuntimeError(
        "selecao_modelos.finalista_escolhido está vazio no YAML: "
        "congele o finalista antes de executar este notebook."
    )

from src.modelos import resolver_finalista as _resolver_finalista

_nomes_yaml = {
    _resolver_finalista(nome): nome
    for nome in FINALISTAS
}

_finalista_canonico = _resolver_finalista(FINALISTA_ESCOLHIDO)

if _finalista_canonico not in _nomes_yaml:
    raise RuntimeError(
        "selecao_modelos.finalista_escolhido="
        f"{FINALISTA_ESCOLHIDO!r} não corresponde a nenhum dos "
        f"finalistas configurados: {FINALISTAS}."
    )

FINALISTA_ESCOLHIDO = _nomes_yaml[_finalista_canonico]

CONFIG_MODELO_HASH = hashlib.sha256(
    json.dumps(
        {
            "models": C.cfg("models"),
            "features": sorted(FEATURES),
            "finalista": _finalista_canonico,
        },
        sort_keys=True,
        default=str,
    ).encode("utf-8")
).hexdigest()[:16]

calibracao_cfg = C.cfg("calibracao") or {}

CALIBRACAO_ATIVA = bool(
    calibracao_cfg.get("ativa", True)
)

METODOS_CANDIDATOS = list(
    calibracao_cfg.get("metodos_candidatos")
    or ["sigmoid", "isotonic"]
)

limiar_cfg = C.cfg("limiar") or {}

LIMIAR_METRICA = limiar_cfg.get("metrica") or "precision"
LIMIAR_MIN_PRECISION = limiar_cfg.get("min_precision")
LIMIAR_MIN_RECALL = limiar_cfg.get("min_recall")

G7_EMPATE_BRIER = 0.0005

print(f"finalistas          : {FINALISTAS}")
print(f"finalista escolhido : {FINALISTA_ESCOLHIDO}")
print(
    "calibração ativa    : "
    f"{CALIBRACAO_ATIVA} ({METODOS_CANDIDATOS})"
)
print(
    "limiar              : selecionado em val_cal e "
    "medido sem reajuste em val_meas"
)
print(
    "regra do limiar     : "
    f"{LIMIAR_METRICA} "
    f"(min_precision={LIMIAR_MIN_PRECISION}, "
    f"min_recall={LIMIAR_MIN_RECALL})"
)

resultados = {}

for nome_yaml in FINALISTAS:
    print(f"\nProcessando finalista: {nome_yaml}")

    pipe = build_finalista(nome_yaml)
    pipe.fit(X_train, y_train)

    prob_base_cal = pipe.predict_proba(X_val_cal)[:, 1]
    prob_base_meas = pipe.predict_proba(X_val_meas)[:, 1]

    brier_base_cal = float(
        brier(y_val_cal, prob_base_cal)
    )
    brier_base_meas = float(
        brier(y_val_meas, prob_base_meas)
    )

    calibracoes = {}

    if CALIBRACAO_ATIVA:
        for metodo in METODOS_CANDIDATOS:
            try:
                if metodo not in {"sigmoid", "isotonic"}:
                    raise ValueError(
                        "Método de calibração inválido: "
                        f"{metodo!r}"
                    )

                calibrador = calibrar(
                    pipe,
                    X_val_cal,
                    y_val_cal,
                    method=metodo,
                )

                prob_cal = calibrador.predict_proba(
                    X_val_cal
                )[:, 1]

                prob_meas = calibrador.predict_proba(
                    X_val_meas
                )[:, 1]

                calibracoes[metodo] = {
                    "pipeline": calibrador,
                    "prob_cal": prob_cal,
                    "prob_meas": prob_meas,
                    "brier_cal": float(
                        brier(y_val_cal, prob_cal)
                    ),
                    "brier_meas": float(
                        brier(y_val_meas, prob_meas)
                    ),
                }

            except Exception as exc:
                print(
                    f"  {nome_yaml}/{metodo}: "
                    f"falhou ({exc!r})"
                )

    if calibracoes:
        # G7 foi decidido por validacao OOF exclusivamente em val_cal.
        # O notebook 10 consome o metodo congelado no YAML; val_meas apenas mede.
        metodo_congelado = calibracao_cfg.get("metodo")

        if metodo_congelado not in calibracoes:
            raise RuntimeError(
                "calibracao.metodo deve estar congelado no YAML e presente "
                f"nos metodos executados. Recebido: {metodo_congelado!r}; "
                f"disponiveis: {sorted(calibracoes)}."
            )

        metodo_escolhido = str(metodo_congelado)
        escolha = {
            "metodo": metodo_escolhido,
            **calibracoes[metodo_escolhido],
        }

    else:
        escolha = {
            "metodo": None,
            "pipeline": pipe,
            "prob_cal": prob_base_cal,
            "prob_meas": prob_base_meas,
            "brier_cal": brier_base_cal,
            "brier_meas": brier_base_meas,
        }

    limiar_escolha = escolher_limiar(
        y_val_cal,
        escolha["prob_cal"],
        metrica=LIMIAR_METRICA,
        min_precision=LIMIAR_MIN_PRECISION,
        min_recall=LIMIAR_MIN_RECALL,
    )

    valor_limiar = float(
        limiar_escolha["limiar"]
    )

    metricas_limiar_cal = metricas_completas(
        y_val_cal,
        escolha["prob_cal"],
        limiar=valor_limiar,
    )

    metricas_limiar_meas = metricas_completas(
        y_val_meas,
        escolha["prob_meas"],
        limiar=valor_limiar,
    )

    resultados[nome_yaml] = {
        "brier_base_cal": brier_base_cal,
        "brier_base_meas": brier_base_meas,
        "prob_base_cal": prob_base_cal,
        "prob_base_meas": prob_base_meas,
        "calibracoes": calibracoes,
        "brier_por_metodo": {
            metodo: float(valores["brier_cal"])
            for metodo, valores in calibracoes.items()
        },
        "escolha": escolha,
        "limiar": limiar_escolha,
        "limiar_escolhido_em": "val_cal",
        "metricas_limiar_cal": metricas_limiar_cal,
        "metricas_limiar_meas": metricas_limiar_meas,
    }

tabela = pd.DataFrame(
    [
        {
            "finalista": nome,
            "metodo_calibracao": resultado[
                "escolha"
            ]["metodo"],
            "brier_base_val_meas": resultado[
                "brier_base_meas"
            ],
            "brier_calibrado_val_meas": resultado[
                "escolha"
            ]["brier_meas"],
            "ganho_brier_val_meas": (
                resultado["brier_base_meas"]
                - resultado["escolha"]["brier_meas"]
            ),
            "limiar_escolhido_val_cal": resultado[
                "limiar"
            ]["limiar"],
            "metrica_limiar": resultado[
                "limiar"
            ]["metrica"],
            "valor_metrica_val_cal": resultado[
                "limiar"
            ]["valor"],
            "recall_val_cal": resultado[
                "metricas_limiar_cal"
            ].get("recall"),
            "precision_val_cal": resultado[
                "metricas_limiar_cal"
            ].get("precision"),
            "recall_val_meas": resultado[
                "metricas_limiar_meas"
            ].get("recall"),
            "precision_val_meas": resultado[
                "metricas_limiar_meas"
            ].get("precision"),
            "f1_val_meas": resultado[
                "metricas_limiar_meas"
            ].get("f1"),
        }
        for nome, resultado in resultados.items()
    ]
)

display(tabela)

tabela_cw = pd.DataFrame()

finalistas          : ['hist_gradient_boosting', 'logistic']
finalista escolhido : hist_gradient_boosting
calibração ativa    : True (['sigmoid', 'isotonic'])
limiar              : selecionado em val_cal e medido sem reajuste em val_meas
regra do limiar     : precision (min_precision=None, min_recall=0.8)

Processando finalista: hist_gradient_boosting



Processando finalista: logistic


,finalista,metodo_calibracao,brier_base_val_meas,brier_calibrado_val_meas,ganho_brier_val_meas,limiar_escolhido_val_cal,metrica_limiar,valor_metrica_val_cal,recall_val_cal,precision_val_cal,recall_val_meas,precision_val_meas,f1_val_meas
0,hist_gradient_boosting,sigmoid,0.063058,0.063063,-0.000005,0.1178,precision,0.252353,0.809817,0.252353,0.809022,0.249082,0.380894
1,logistic,sigmoid,0.065028,0.065043,-0.000014,0.0982,precision,0.243905,0.801272,0.243905,0.800079,0.241164,0.370616


## Comparação de `class_weight`

O `class_weight="balanced"` desloca as probabilidades brutas para cima da prevalência real (8,4% na coorte definitiva). A calibração em `val_cal` corrige — mas ainda não foi medido se **treinar sem peso** entrega Brier/ECE iguais ou melhores com PR-AUC comparável.

Este bloco treina **as duas variantes com os mesmos hiperparâmetros** (os do YAML vencedor) e o **mesmo método de calibração** já escolhido pelo C_5, mede em `val_meas` e mostra o comparativo. Não altera a escolha do C_5 — apenas documenta o efeito e alimenta a decisão do item.

In [4]:
if CALIBRACAO_ATIVA:
    comparacao_class_weight = []

    for nome_yaml in FINALISTAS:
        metodo = resultados[
            nome_yaml
        ]["escolha"]["metodo"]

        if metodo is None:
            print(
                f"  {nome_yaml}: sem calibrador válido; "
                "comparação de class_weight ignorada."
            )
            continue

        for class_weight in (None, "balanced"):
            pipe_cw = build_finalista(
                nome_yaml,
                class_weight=class_weight,
            )

            pipe_cw.fit(
                X_train,
                y_train,
            )

            calibrador_cw = calibrar(
                pipe_cw,
                X_val_cal,
                y_val_cal,
                method=metodo,
            )

            prob_cw = calibrador_cw.predict_proba(
                X_val_meas
            )[:, 1]

            metricas_cw = metricas_completas(
                y_val_meas,
                prob_cw,
                limiar=0.5,
            )

            comparacao_class_weight.append(
                {
                    "finalista": nome_yaml,
                    "metodo": metodo,
                    "class_weight": (
                        "sem_peso"
                        if class_weight is None
                        else class_weight
                    ),
                    "brier": metricas_cw["brier"],
                    "pr_auc": metricas_cw["pr_auc"],
                    "roc_auc": metricas_cw["auc"],
                    "ece": float(
                        ece(
                            y_val_meas,
                            prob_cw,
                            n_bins=10,
                        )
                    ),
                }
            )

    if comparacao_class_weight:
        tabela_cw = (
            pd.DataFrame(
                comparacao_class_weight
            )
            .sort_values(
                ["finalista", "brier"],
                ascending=[True, True],
            )
            .reset_index(drop=True)
        )

        display(tabela_cw)

        veredito_cw = (
            tabela_cw
            .sort_values(
                ["finalista", "brier"],
                ascending=[True, True],
            )
            .groupby(
                "finalista",
                as_index=False,
                sort=False,
            )
            .head(1)
            [
                [
                    "finalista",
                    "class_weight",
                    "brier",
                    "pr_auc",
                ]
            ]
            .reset_index(drop=True)
        )

        print(
            "\nVeredito "
            "(menor Brier pós-calibração):"
        )

        display(veredito_cw)

    else:
        tabela_cw = pd.DataFrame()

        print(
            "Nenhuma comparação de class_weight "
            "foi produzida."
        )

else:
    tabela_cw = pd.DataFrame()

    print(
        "Calibração desativada; comparação de "
        "class_weight não executada."
    )

,finalista,metodo,class_weight,brier,pr_auc,roc_auc,ece
0,hist_gradient_boosting,sigmoid,sem_peso,0.063063,0.362670,0.866201,0.002423
1,hist_gradient_boosting,sigmoid,balanced,0.063268,0.357242,0.865695,0.003761
2,logistic,sigmoid,sem_peso,0.065043,0.323972,0.854423,0.005921
3,logistic,sigmoid,balanced,0.065145,0.321635,0.854427,0.005382



Veredito (menor Brier pós-calibração):


,finalista,class_weight,brier,pr_auc
0,hist_gradient_boosting,sem_peso,0.063063,0.362670
1,logistic,sem_peso,0.065043,0.323972


## Diagnósticos e persistência

Curva de confiabilidade, decomposição de Brier, ECE e MCE em
`val_meas` com o método escolhido. Persistimos tudo em `outputs/`.

In [5]:
diagnosticos = {}
linhas_diag  = []

for nome, r in resultados.items():
    prob = r["escolha"]["prob_meas"]
    d = {
        "curva":  curva_confiabilidade(y_val_meas, prob, n_bins=10),
        "decomp": brier_decomposicao(y_val_meas, prob, n_bins=10),
        "ece":    float(ece(y_val_meas, prob, n_bins=10)),
        "mce":    float(mce(y_val_meas, prob, n_bins=10)),
    }
    diagnosticos[nome] = d
    linhas_diag.append({
        "finalista":   nome,
        "metodo":      r["escolha"]["metodo"] or "sem_calibracao",
        "ece":         d["ece"],
        "mce":         d["mce"],
        "brier":       d["decomp"]["brier"],
        "reliability": d["decomp"]["reliability"],
        "resolution":  d["decomp"]["resolution"],
    })

display(pd.DataFrame(linhas_diag))

,finalista,metodo,ece,mce,brier,reliability,resolution
0,hist_gradient_boosting,sigmoid,0.002423,0.084307,0.063063,0.000022,0.01352
1,logistic,sigmoid,0.005921,0.205630,0.065043,0.000273,0.01175


In [6]:
timestamp = pd.Timestamp.now(
    tz="UTC"
).strftime("%Y%m%dT%H%M%SZ")

prefixo = (
    "10_calibracao_limiar_"
    f"{manifest['cohort_kind']}"
)

caminho_tabela = (
    TABLES_DIR
    / f"{prefixo}_escolhas.csv"
)

caminho_cw = (
    TABLES_DIR
    / f"{prefixo}_comparacao_class_weight.csv"
)

caminho_predicoes_cal = (
    PREDICTIONS_DIR
    / f"{prefixo}_predicoes_val_cal.parquet"
)

caminho_predicoes_meas = (
    PREDICTIONS_DIR
    / f"{prefixo}_predicoes_val_meas.parquet"
)

caminho_manifesto = (
    TABLES_DIR
    / f"{prefixo}_manifesto.json"
)

tabela.to_csv(
    caminho_tabela,
    index=False,
    encoding="utf-8-sig",
)

if not tabela_cw.empty:
    tabela_cw.to_csv(
        caminho_cw,
        index=False,
        encoding="utf-8-sig",
    )

artefatos_diag = {}

for nome, diagnostico in diagnosticos.items():
    caminho_curva = (
        TABLES_DIR
        / f"{prefixo}_{nome}_curva.csv"
    )

    caminho_limiar = (
        TABLES_DIR
        / f"{prefixo}_{nome}_limiar_val_cal.csv"
    )

    diagnostico["curva"].to_csv(
        caminho_curva,
        index=False,
        encoding="utf-8-sig",
    )

    resultados[nome]["limiar"]["tabela"].to_csv(
        caminho_limiar,
        index=False,
        encoding="utf-8-sig",
    )

    artefatos_diag[nome] = {
        "curva_val_meas": str(caminho_curva),
        "tabela_limiar_val_cal": str(
            caminho_limiar
        ),
        "ece_val_meas": diagnostico["ece"],
        "mce_val_meas": diagnostico["mce"],
        "brier_decomp_val_meas": {
            chave: float(valor)
            for chave, valor
            in diagnostico["decomp"].items()
        },
    }

predicoes_cal = []
predicoes_meas = []

for nome, resultado in resultados.items():
    limiar_modelo = float(
        resultado["limiar"]["limiar"]
    )

    predicoes_cal.append(
        pd.DataFrame(
            {
                "y_true": y_val_cal.to_numpy(),
                "y_prob": resultado[
                    "escolha"
                ]["prob_cal"],
                "y_pred": (
                    resultado["escolha"]["prob_cal"]
                    >= limiar_modelo
                ).astype(int),
                "modelo": (
                    f"{nome}_"
                    f"{resultado['escolha']['metodo']}"
                ),
                "particao_avaliacao": "val_cal",
                "limiar": limiar_modelo,
                "cohort_kind": manifest[
                    "cohort_kind"
                ],
            }
        )
    )

    predicoes_meas.append(
        pd.DataFrame(
            {
                "y_true": y_val_meas.to_numpy(),
                "y_prob": resultado[
                    "escolha"
                ]["prob_meas"],
                "y_pred": (
                    resultado["escolha"]["prob_meas"]
                    >= limiar_modelo
                ).astype(int),
                "modelo": (
                    f"{nome}_"
                    f"{resultado['escolha']['metodo']}"
                ),
                "particao_avaliacao": "val_meas",
                "limiar": limiar_modelo,
                "cohort_kind": manifest[
                    "cohort_kind"
                ],
            }
        )
    )

pred_val_cal = pd.concat(
    predicoes_cal,
    ignore_index=True,
)

pred_val_meas = pd.concat(
    predicoes_meas,
    ignore_index=True,
)

pred_val_cal.to_parquet(
    caminho_predicoes_cal,
    index=False,
)

pred_val_meas.to_parquet(
    caminho_predicoes_meas,
    index=False,
)

manifesto_c5 = {
    "cohort_kind": manifest["cohort_kind"],
    "dataset_sha256": manifest["dataset_sha256"],
    "feature_columns": FEATURES,
    "target_column": ALVO,
    "partition_column": PARTICAO,
    "n_treino": int(len(treino)),
    "n_validacao": int(len(validacao)),
    "n_val_cal": int(len(X_val_cal)),
    "n_val_meas": int(len(X_val_meas)),
    "n_teste_fechado": n_teste,
    "honest_calib_split": bool(
        HONEST_CALIB_SPLIT
    ),
    "calib_split_frac": float(
        CALIB_SPLIT_FRAC
    ),
    "calib_split_seed": int(
        CALIB_SPLIT_SEED
    ),
    "calib_split_hash": CALIB_SPLIT_HASH,
    "finalistas": FINALISTAS,
    "finalista_escolhido": (
        FINALISTA_ESCOLHIDO
    ),
    "config_modelo_hash": (
        CONFIG_MODELO_HASH
    ),
    "calibracao_ativa": (
        CALIBRACAO_ATIVA
    ),
    "metodos_candidatos": (
        METODOS_CANDIDATOS
    ),
    "g7_empate_brier": (
        G7_EMPATE_BRIER
    ),
    "limiar_metrica": (
        LIMIAR_METRICA
    ),
    "limiar_min_precision": (
        LIMIAR_MIN_PRECISION
    ),
    "limiar_min_recall": (
        LIMIAR_MIN_RECALL
    ),
    "limiar_escolhido_em": "val_cal",
    "limiar_medido_em": "val_meas",
    "limiar_reajustado_em_val_meas": False,
    "escolhas": {
        nome: {
            "metodo": resultado[
                "escolha"
            ]["metodo"],
            "brier_base_val_meas": resultado[
                "brier_base_meas"
            ],
            "brier_calibrado_val_meas": resultado[
                "escolha"
            ]["brier_meas"],
            "brier_por_metodo_val_cal": (
                resultado["brier_por_metodo"]
            ),
            "limiar": resultado[
                "limiar"
            ]["limiar"],
            "limiar_metrica": resultado[
                "limiar"
            ]["metrica"],
            "metricas_val_cal": resultado[
                "metricas_limiar_cal"
            ],
            "metricas_val_meas": resultado[
                "metricas_limiar_meas"
            ],
        }
        for nome, resultado
        in resultados.items()
    },
    "comparacao_class_weight": (
        tabela_cw.to_dict(
            orient="records"
        )
        if not tabela_cw.empty
        else []
    ),
    "diagnosticos": artefatos_diag,
    "g7_status": G7_STATUS,
    "g8_status": G8_STATUS,
    "teste_aberto": False,
    "yaml_atualizado": False,
    "artefatos": {
        "tabela_escolhas": str(
            caminho_tabela
        ),
        "comparacao_class_weight": (
            str(caminho_cw)
            if not tabela_cw.empty
            else None
        ),
        "predicoes_val_cal": str(
            caminho_predicoes_cal
        ),
        "predicoes_val_meas": str(
            caminho_predicoes_meas
        ),
    },
}

caminho_manifesto.write_text(
    json.dumps(
        manifesto_c5,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print(f"Tabela       : {caminho_tabela}")

if not tabela_cw.empty:
    print(f"Class_weight : {caminho_cw}")

print(
    f"Predições cal: {caminho_predicoes_cal}"
)
print(
    f"Predições mea: {caminho_predicoes_meas}"
)
print(f"Manifesto    : {caminho_manifesto}")

Tabela       : <REPO>\outputs\tables\10_calibracao_limiar_final_escolhas.csv
Class_weight : <REPO>\outputs\tables\10_calibracao_limiar_final_comparacao_class_weight.csv
Predições cal: <REPO>\outputs\predictions\10_calibracao_limiar_final_predicoes_val_cal.parquet
Predições mea: <REPO>\outputs\predictions\10_calibracao_limiar_final_predicoes_val_meas.parquet
Manifesto    : <REPO>\outputs\tables\10_calibracao_limiar_final_manifesto.json


In [7]:
melhor_nome = FINALISTA_ESCOLHIDO

metodo_escolhido = resultados[
    melhor_nome
]["escolha"]["metodo"]

limiar_escolhido = resultados[
    melhor_nome
]["limiar"]

metricas_escolha = resultados[
    melhor_nome
]["metricas_limiar_cal"]

metricas_medicao = resultados[
    melhor_nome
]["metricas_limiar_meas"]

cw_recomendado_por_finalista = {}

if not tabela_cw.empty:
    for nome in FINALISTAS:
        subconjunto = tabela_cw[
            tabela_cw["finalista"] == nome
        ]

        if subconjunto.empty:
            continue

        melhor_cw = (
            subconjunto
            .sort_values(
                "brier",
                ascending=True,
            )
            .iloc[0]
        )

        cw_recomendado_por_finalista[
            nome
        ] = melhor_cw["class_weight"]

linhas_proposta = [
    "# Proposta de atualização do YAML.",
    "# Não aplicada automaticamente.",
    f"# cohort_kind: {manifest['cohort_kind']}",
    f"# G7/G8 antes da atualização: "
    f"{G7_STATUS}/{G8_STATUS}",
    f"# finalista congelado: {melhor_nome}",
    "# limiar selecionado em val_cal",
    "# limiar medido sem reajuste em val_meas",
]

if cw_recomendado_por_finalista:
    linhas_proposta.append(
        "# class_weight recomendado:"
    )

    for nome, class_weight in (
        cw_recomendado_por_finalista.items()
    ):
        linhas_proposta.append(
            f"#   {nome}: {class_weight}"
        )

linhas_proposta += [
    "calibracao:",
    f"  metodo: {metodo_escolhido}",
    "limiar:",
    "  otimo: "
    f"{float(limiar_escolhido['limiar']):.4f}",
    "  metrica: "
    f"{limiar_escolhido['metrica']}",
    "  min_recall: "
    f"{LIMIAR_MIN_RECALL}",
    "# Resultado no conjunto de escolha:",
    "#   recall_val_cal: "
    f"{metricas_escolha.get('recall')}",
    "#   precision_val_cal: "
    f"{metricas_escolha.get('precision')}",
    "# Resultado no conjunto de medida:",
    "#   recall_val_meas: "
    f"{metricas_medicao.get('recall')}",
    "#   precision_val_meas: "
    f"{metricas_medicao.get('precision')}",
]

proposta_yaml = "\n".join(
    linhas_proposta
)

print(proposta_yaml)

with Run(
    modelo="10_calibracao_limiar",
    parametros={
        "cohort_kind": manifest[
            "cohort_kind"
        ],
        "finalistas": FINALISTAS,
        "calibracao_ativa": (
            CALIBRACAO_ATIVA
        ),
        "metodos_candidatos": (
            METODOS_CANDIDATOS
        ),
        "honest_calib_split": (
            HONEST_CALIB_SPLIT
        ),
        "limiar_selecionado_em": "val_cal",
        "limiar_medido_em": "val_meas",
        "limiar_reajustado_em_val_meas": (
            False
        ),
        "limiar_metrica": LIMIAR_METRICA,
        "min_precision": (
            LIMIAR_MIN_PRECISION
        ),
        "min_recall": LIMIAR_MIN_RECALL,
        "teste_aberto": False,
        "yaml_atualizado": False,
        "comparacao_class_weight": (
            tabela_cw.to_dict(
                orient="records"
            )
            if not tabela_cw.empty
            else []
        ),
        "cw_recomendado_por_finalista": (
            cw_recomendado_por_finalista
        ),
    },
    dataset={
        "n_treino": int(len(treino)),
        "n_val_cal": int(len(X_val_cal)),
        "n_val_meas": int(len(X_val_meas)),
        "n_teste_fechado": n_teste,
    },
    features={
        "n_features": len(FEATURES),
        "usadas": FEATURES,
    },
    notas=(
        "C5: calibrador ajustado em val_cal; "
        "método congelado após validação OOF em val_cal; val_meas usado somente para medição; "
        "limiar selecionado em val_cal e "
        "medido sem reajuste em val_meas. "
        "Teste não consultado."
    ),
    dataset_path=Path(
        manifest["dataset_path"]
    ),
    dataset_sha256=manifest[
        "dataset_sha256"
    ],
) as run:
    for nome, resultado in resultados.items():
        run.metrica(
            **{
                f"{nome}_brier_base_val_meas": float(
                    resultado[
                        "brier_base_meas"
                    ]
                ),
                f"{nome}_brier_calibrado_val_meas": float(
                    resultado[
                        "escolha"
                    ]["brier_meas"]
                ),
                f"{nome}_limiar": float(
                    resultado[
                        "limiar"
                    ]["limiar"]
                ),
                f"{nome}_recall_val_cal": float(
                    resultado[
                        "metricas_limiar_cal"
                    ]["recall"]
                ),
                f"{nome}_precision_val_cal": float(
                    resultado[
                        "metricas_limiar_cal"
                    ]["precision"]
                ),
                f"{nome}_recall_val_meas": float(
                    resultado[
                        "metricas_limiar_meas"
                    ]["recall"]
                ),
                f"{nome}_precision_val_meas": float(
                    resultado[
                        "metricas_limiar_meas"
                    ]["precision"]
                ),
            }
        )

    if not tabela_cw.empty:
        for _, linha in tabela_cw.iterrows():
            sufixo = (
                f"{linha['finalista']}_"
                f"cw_{linha['class_weight']}"
            )

            run.metrica(
                **{
                    f"{sufixo}_brier": float(
                        linha["brier"]
                    ),
                    f"{sufixo}_pr_auc": float(
                        linha["pr_auc"]
                    ),
                    f"{sufixo}_ece": float(
                        linha["ece"]
                    ),
                }
            )

print("Run registrado em:", run.pasta)

# Proposta de atualização do YAML.
# Não aplicada automaticamente.
# cohort_kind: final
# G7/G8 antes da atualização: aprovado/aprovado
# finalista congelado: hist_gradient_boosting
# limiar selecionado em val_cal
# limiar medido sem reajuste em val_meas
# class_weight recomendado:
#   hist_gradient_boosting: sem_peso
#   logistic: sem_peso
calibracao:
  metodo: sigmoid
limiar:
  otimo: 0.1178
  metrica: precision
  min_recall: 0.8
# Resultado no conjunto de escolha:
#   recall_val_cal: 0.8098171701112877
#   precision_val_cal: 0.2523532325984642
# Resultado no conjunto de medida:
#   recall_val_meas: 0.8090222575516693
#   precision_val_meas: 0.24908223201174742
Execução registrada em: <REPO>\outputs\runs\20261105T201249483681Z_10_calibracao_limiar
Run registrado em: <REPO>\outputs\runs\20261105T201249483681Z_10_calibracao_limiar
